# Staging & Fact Table Construction

## Objective

This notebook performs:

- Data cleaning and preprocessing
- Validation of order status
- Revenue aggregation
- Join operations
- Construction of the `fact_orders` table
- Grain validation

This represents the transformation layer of the analytics workflow.

## Import & Load

In [3]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

## Staging Layer

### Parse Date Columns

In [4]:
date_cols_orders = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols_orders:
    orders[col] = pd.to_datetime(orders[col])

### Filter Valid Orders

In [5]:
orders = orders[orders["order_status"] == "delivered"].copy()

### Revenue Aggregation

In [6]:
# Agrupar payments antes de unir
order_revenue = (
    payments
    .groupby("order_id", as_index=False)
    .agg({"payment_value": "sum"})
    .rename(columns={"payment_value": "order_value"})
)

### Review Selection

In [7]:
# Si hay multiples reviews por orden, tomar el promedio
order_reviews = (
    reviews
    .groupby("order_id", as_index=False)
    .agg({"review_score": "mean"})
)

## FACT CONSTRUCTION

### Join Customers

In [11]:
fact_orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

### Join Revenue

In [12]:
fact_orders = fact_orders.merge(
    order_revenue,
    on="order_id",
    how="left"  
)

### Join Reviews

In [13]:
fact_orders = fact_orders.merge(
    order_reviews,
    on="order_id",
    how="left"
)

## FEATURE ENGINEERING BÁSICO

### Delivery Delay

In [15]:
fact_orders["delivery_delay_days"] = (
    fact_orders["order_delivered_customer_date"] -
    fact_orders["order_estimated_delivery_date"]
).dt.days

### Year & Month Columns

In [16]:
fact_orders["order_year"] = fact_orders["order_purchase_timestamp"].dt.year
fact_orders["order_month"] = fact_orders["order_purchase_timestamp"].dt.month

## GRAIN VALIDATION

### Grain Check

In [18]:
print("Total rows:", len(fact_orders))
print("Unique order_id:", fact_orders["order_id"].nunique())

Total rows: 96478
Unique order_id: 96478


### Grain Validation Observations

The number of rows matches the number of unique `order_id` values.

This confirms that the fact table maintains the intended grain:

**1 row per order**

No unintended duplication occurred during joins.

## Sanity Checks

### Null Checks

In [19]:
fact_orders.isnull().sum()

order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 14
order_delivered_carrier_date       2
order_delivered_customer_date      8
order_estimated_delivery_date      0
customer_unique_id                 0
order_value                        1
review_score                     646
delivery_delay_days                8
order_year                         0
order_month                        0
dtype: int64

In [27]:
# Verificar si son delivered y no tienen fecha de entrega
fact_orders[fact_orders["order_delivered_customer_date"].isna()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,order_value,review_score,delivery_delay_days,order_year,order_month
2921,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18,13467e882eb3a701826435ee4424f2bd,134.83,5.0,NaN,2017,11
20021,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16,2f17c5b324ad603491521b279a9ff4de,354.24,5.0,NaN,2018,6
42530,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30,1bd06a0c0df8b23dacfd3725d2dc0bb9,158.07,5.0,NaN,2018,7
76909,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30,3bc508d482a402715be4d5cf4020cc81,158.07,5.0,NaN,2018,7
80392,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24,ebf7e0d43a78c81991a4c59c145c75db,204.62,5.0,NaN,2018,7
89883,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaT,2017-06-23,d77cf4be2654aa70ef150f8bfec076a6,194.00,5.0,NaN,2017,5
94731,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26,cce5e8188bf42ffb3bb5b18ff58f5965,120.12,1.0,NaN,2018,6
95117,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19,175378436e2978be55b8f4316bce4811,54.97,5.0,NaN,2018,6


In [28]:
# Entonces las eliminamos
fact_orders = fact_orders.dropna(subset=["order_delivered_customer_date"])

In [29]:
# Verificar si la diferencia con purchase date es mínima
fact_orders[fact_orders["order_approved_at"].isna()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,order_value,review_score,delivery_delay_days,order_year,order_month
5171,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17,8a9a08c7ca8900a200d83cf838a07e0b,349.01,4.0,-16.0,2017,2
16098,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21,91efb7fcabc17925099dced52435837f,396.86,5.0,-19.0,2017,2
18494,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17,e1f01a1bd6485e58ad3c769a5427d8a8,65.52,5.0,-16.0,2017,2
21999,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31,7e1a5ca61b572d76b64b6688b9f96473,106.81,5.0,-22.0,2017,2
22478,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20,c8822fce1d0bfa7ddf0da24fff947172,95.76,5.0,-18.0,2017,2
26014,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01,6ff8b0d7b35d5c945633b8d60165691b,54.51,4.0,-30.0,2017,1
37158,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27,2e0a2166aa23da2472c6a60c4af6f7a6,163.43,5.0,-25.0,2017,2
38171,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22,c4c0011e639bdbcf26059ddc38bd3c18,39.95,5.0,-20.0,2017,2
46957,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaN,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16,d49f3dae6bad25d05160fc17aca5942d,60.42,2.0,-38.0,2017,1
59912,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaN,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20,5a4fa4919cbf2b049e72be460a380e5b,154.23,5.0,-17.0,2017,2


In [30]:
# Como no son pocas filas, vamos a reemplazar la fecha de aprobación con la fecha de compra
# porque la aprobación suele ocurrir muy cerca de la compra, y no queremos perder esos datos
fact_orders["order_approved_at"] = fact_orders["order_approved_at"].fillna(fact_orders["order_purchase_timestamp"])

In [31]:
# Los nulls del review score los dejamos así, porque no sabemos si el cliente no dejó review o si no se le pidió.
# El null del order value lo eliminamos, porque al ser solo 1 fila, tiene impacto mínimo.
fact_orders = fact_orders.dropna(subset=["order_value"])

In [34]:
# El null del order delivered carrier date lo eliminamos, porque al ser solo 1 fila, tiene impacto mínimo.
fact_orders = fact_orders.dropna(subset=["order_delivered_carrier_date"])

### Revenue Distribution Check

In [35]:
fact_orders["order_value"].describe()

count    96468.000000
mean       159.854966
std        218.822041
min          9.590000
25%         61.880000
50%        105.280000
75%        176.330000
max      13664.080000
Name: order_value, dtype: float64

In [39]:
# Ultima verificación de unicidad de order_id
fact_orders["order_id"].nunique() == len(fact_orders)
# Exportar fact_orders a csv
fact_orders.to_csv("../data/processed/fact_orders.csv", index=False)

## Summary

In this notebook:

- Filtered delivered orders
- Aggregated revenue at order level
- Merged customer identity using `customer_unique_id`
- Incorporated review scores
- Calculated delivery delay
- Added temporal columns
- Validated grain consistency
- Minor missing delivery dates (8 rows) removed due to inconsistency.
- 14 missing approval timestamps filled with purchase timestamp.
- 1 null order_value removed.
- 646 null review scores preserved as non-response cases.

The resulting `fact_orders` table is ready for customer-level aggregation and retention analysis.